# Scraper Infarmed PMRO (Playwright)

Notebook para extrair a tabela de doações do PMRO, com:
- seleção de ano
- paginação completa (inclui `...`)
- gravação incremental (JSONL + CSV + checkpoint)
- retoma (`resume`) em caso de falha


In [1]:
# Se precisares de preparar o ambiente (executa apenas uma vez):
# !uv venv .venv
# !uv pip install --python .venv/bin/python playwright
# !.venv/bin/python -m playwright install chromium


In [2]:
from __future__ import annotations

import csv
import json
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from playwright.sync_api import Page, TimeoutError as PlaywrightTimeoutError, sync_playwright

URL = 'https://extranet.infarmed.pt/pmro/Publico/ListagemPublica.aspx'
TABLE_SELECTOR = '#ctl00_ContentPlaceHolder1_gvDoacoesPublic'
YEAR_BUTTONS_SELECTOR = "#ctl00_ContentPlaceHolder1_painel_anos1 input[id^='ctl00_ContentPlaceHolder1_1_']"
PAGER_LINKS_SELECTOR = "a[id^='ctl00_ContentPlaceHolder1_pageCounterDoacoes_'][id$='_Pager']"
SELECTED_PAGE_SELECTOR = "a.pager-class-selected[id^='ctl00_ContentPlaceHolder1_pageCounterDoacoes_'][id$='_Pager']"
ELLIPSIS_REGEX = re.compile(r'^\s*\.\.\.\s*$')


In [3]:
# Parametros da execução
YEAR = 2025                   # ano a extrair
HEADLESS = True               # True = sem abrir janela
MAX_PAGES = None              # None = extrai tudo
RESUME = True                 # True = retoma de checkpoint se existir

BASE_DIR = Path('infarmed_output')
RUN_NAME = f'doacoes_{YEAR}'
JSONL_PATH = BASE_DIR / f'{RUN_NAME}.jsonl'
CSV_PATH = BASE_DIR / f'{RUN_NAME}.csv'
CHECKPOINT_PATH = BASE_DIR / f'{RUN_NAME}_checkpoint.json'
FINAL_JSON_PATH = BASE_DIR / f'{RUN_NAME}.json'

BASE_DIR.mkdir(parents=True, exist_ok=True)
JSONL_PATH, CSV_PATH, CHECKPOINT_PATH, FINAL_JSON_PATH


(PosixPath('infarmed_output/doacoes_2025.jsonl'),
 PosixPath('infarmed_output/doacoes_2025.csv'),
 PosixPath('infarmed_output/doacoes_2025_checkpoint.json'),
 PosixPath('infarmed_output/doacoes_2025.json'))

In [4]:
def get_available_years(page: Page) -> list[int]:
    ids = page.locator(YEAR_BUTTONS_SELECTOR).evaluate_all(
        "els => els.map(e => e.id)"
    )
    years: list[int] = []
    for element_id in ids:
        m = re.search(r'(\d{4})$', element_id)
        if m:
            years.append(int(m.group(1)))
    return sorted(set(years), reverse=True)


def select_year(page: Page, year: int) -> None:
    selector = f'#ctl00_ContentPlaceHolder1_1_{year}'
    if page.locator(selector).count() == 0:
        available = get_available_years(page)
        raise ValueError(f'Ano {year} nao disponivel. Anos disponiveis: {available}')

    page.locator(selector).first.click()
    page.wait_for_function(
        "(sel) => document.querySelector(sel)?.className.includes('btngeral_hover')",
        arg=selector,
        timeout=60000,
    )
    page.wait_for_selector(TABLE_SELECTOR, timeout=120000)


def get_selected_page(page: Page) -> int:
    text = page.locator(SELECTED_PAGE_SELECTOR).first.inner_text().strip()
    m = re.search(r'\d+', text)
    if not m:
        raise RuntimeError(f'Nao foi possivel ler pagina selecionada: {text!r}')
    return int(m.group(0))


def get_headers(page: Page) -> list[str]:
    headers = page.eval_on_selector(
        TABLE_SELECTOR,
        """(table) =>
            Array.from(table.querySelectorAll('tr th'))
                .map(th => (th.textContent || '').replace(/\s+/g, ' ').trim())
        """,
    )
    return [h for h in headers if h]


def extract_rows(page: Page, headers: list[str], year: int, page_number: int) -> list[dict[str, Any]]:
    raw_rows = page.eval_on_selector(
        TABLE_SELECTOR,
        """(table) => {
            const rows = [];
            const trs = Array.from(table.querySelectorAll('tr'));
            for (const tr of trs) {
                const tds = Array.from(tr.querySelectorAll('td'));
                if (!tds.length) continue;
                rows.push(tds.map(td => (td.textContent || '').replace(/\s+/g, ' ').trim()));
            }
            return rows;
        }""",
    )

    records: list[dict[str, Any]] = []
    for idx, row in enumerate(raw_rows, start=1):
        item: dict[str, Any] = {
            'ano': year,
            'pagina': page_number,
            'linha_pagina': idx,
        }
        for col_idx, header in enumerate(headers):
            item[header] = row[col_idx] if col_idx < len(row) else ''
        records.append(item)
    return records


def wait_for_selected_page(page: Page, target_page: int, timeout: int = 30000) -> bool:
    try:
        page.wait_for_function(
            """({selector, targetPage}) => {
                const el = document.querySelector(selector);
                if (!el) return false;
                const current = parseInt((el.textContent || '').trim(), 10);
                return !Number.isNaN(current) && current === targetPage;
            }""",
            arg={'selector': SELECTED_PAGE_SELECTOR, 'targetPage': target_page},
            timeout=timeout,
        )
        return True
    except PlaywrightTimeoutError:
        return False


def wait_for_selected_page_change(page: Page, previous_page: int, timeout: int = 30000) -> bool:
    try:
        page.wait_for_function(
            """({selector, previousPage}) => {
                const el = document.querySelector(selector);
                if (!el) return false;
                const current = parseInt((el.textContent || '').trim(), 10);
                return !Number.isNaN(current) && current !== previousPage;
            }""",
            arg={'selector': SELECTED_PAGE_SELECTOR, 'previousPage': previous_page},
            timeout=timeout,
        )
        return True
    except PlaywrightTimeoutError:
        return False


def wait_for_update_progress_idle(page: Page, timeout: int = 45000) -> bool:
    # O site usa update panel ASP.NET; enquanto o overlay estiver ativo, o click no pager falha.
    try:
        page.wait_for_function(
            """() => {
                const el = document.querySelector('#ctl00_updateProgress');
                if (!el) return true;
                const style = window.getComputedStyle(el);
                const ariaHidden = (el.getAttribute('aria-hidden') || '').toLowerCase() === 'true';
                const hiddenByStyle = style.display === 'none' || style.visibility === 'hidden' || Number(style.opacity || '1') === 0;
                const rect = el.getBoundingClientRect();
                const hasArea = rect.width > 0 && rect.height > 0;
                return ariaHidden || hiddenByStyle || !hasArea;
            }""",
            timeout=timeout,
        )
        return True
    except PlaywrightTimeoutError:
        return False


def resilient_click(locator, page: Page, attempts: int = 4, click_timeout: int = 7000) -> bool:
    for attempt in range(1, attempts + 1):
        wait_for_update_progress_idle(page, timeout=45000)

        try:
            locator.first.click(timeout=click_timeout)
            wait_for_update_progress_idle(page, timeout=45000)
            return True
        except PlaywrightTimeoutError:
            if attempt == attempts:
                return False
            page.wait_for_timeout(min(250 * attempt, 1000))

    return False


def get_visible_page_numbers(page: Page) -> list[int]:
    texts = page.locator(PAGER_LINKS_SELECTOR).all_inner_texts()
    nums: list[int] = []
    for text in texts:
        value = text.strip()
        if value.isdigit():
            nums.append(int(value))
    return sorted(set(nums))


def click_page_number(page: Page, page_number: int) -> bool:
    locator = page.locator(PAGER_LINKS_SELECTOR).filter(has_text=re.compile(rf'^\s*{page_number}\s*$'))
    if locator.count() == 0:
        return False

    if not resilient_click(locator, page):
        return False

    return wait_for_selected_page(page, page_number)


def click_last_visible_page(page: Page, current_page: int, target_page: int) -> bool:
    candidates = [n for n in get_visible_page_numbers(page) if current_page < n <= target_page]
    if not candidates:
        return False

    jump_page = max(candidates)
    if jump_page <= current_page:
        return False

    return click_page_number(page, jump_page)


def click_forward_ellipsis(page: Page, current_page: int) -> bool:
    # Em algumas situações o primeiro clique nao muda a pagina; tentamos algumas vezes.
    for _ in range(3):
        ellipsis = page.locator(PAGER_LINKS_SELECTOR).filter(has_text=ELLIPSIS_REGEX)
        if ellipsis.count() == 0:
            return False

        if not resilient_click(ellipsis.last, page):
            continue

        if wait_for_selected_page_change(page, current_page, timeout=25000):
            return True

    return False


def go_to_page(page: Page, target_page: int, log_every: int = 25) -> bool:
    if target_page <= 1:
        return True

    safety_steps = max(300, target_page + 50)
    steps = 0
    stagnant_steps = 0

    while steps < safety_steps:
        steps += 1
        current_page = get_selected_page(page)

        if current_page == target_page:
            return True
        if current_page > target_page:
            return False

        if steps == 1 or steps % log_every == 0:
            print(f'[RESUME] a navegar... pagina atual={current_page}, alvo={target_page}')

        move_label = ''

        # Ordem de tentativa: alvo direto -> maior numero visivel (salto) -> +1 -> ...
        moved = click_page_number(page, target_page)
        if moved:
            move_label = 'alvo direto'

        if not moved:
            visible_before = get_visible_page_numbers(page)
            candidates = [n for n in visible_before if current_page < n <= target_page]
            jump_target = max(candidates) if candidates else None
            moved = click_last_visible_page(page, current_page, target_page)
            if moved:
                move_label = f'ultimo visivel ({jump_target})'

        if not moved:
            moved = click_page_number(page, current_page + 1)
            if moved:
                move_label = f'+1 ({current_page + 1})'

        if not moved:
            moved = click_forward_ellipsis(page, current_page)
            if moved:
                move_label = 'reticencias (...)'

        if not moved:
            print(f'[RESUME] sem movimento na paginacao a partir da pagina {current_page}.')
            return False

        new_page = get_selected_page(page)
        print(f'[RESUME] salto: {current_page} -> {new_page} via {move_label}')

        if new_page <= current_page:
            stagnant_steps += 1
            if stagnant_steps >= 3:
                print(f'[RESUME] paginacao estagnada perto da pagina {current_page}.')
                return False
        else:
            stagnant_steps = 0

    print(f'[RESUME] limite de seguranca atingido ao tentar ir para pagina {target_page}.')
    return False


<>:39: SyntaxWarning: invalid escape sequence '\s'
<>:50: SyntaxWarning: invalid escape sequence '\s'
<>:39: SyntaxWarning: invalid escape sequence '\s'
<>:50: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_171494/1292588376.py:39: SyntaxWarning: invalid escape sequence '\s'
  """(table) =>
/tmp/ipykernel_171494/1292588376.py:50: SyntaxWarning: invalid escape sequence '\s'
  """(table) => {


In [5]:
def reset_outputs(*paths: Path) -> None:
    for path in paths:
        if path.exists():
            path.unlink()


def load_checkpoint(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {'visited_pages': [], 'total_rows': 0}
    return json.loads(path.read_text(encoding='utf-8'))


def save_checkpoint(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')


def ensure_csv_header(path: Path, fieldnames: list[str]) -> None:
    if path.exists() and path.stat().st_size > 0:
        return
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()


def append_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    if not rows:
        return
    with path.open('a', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def append_csv(path: Path, rows: list[dict[str, Any]], fieldnames: list[str]) -> None:
    if not rows:
        return
    with path.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writerows(rows)


def export_json_array_from_jsonl(jsonl_path: Path, json_path: Path) -> None:
    data: list[dict[str, Any]] = []
    if jsonl_path.exists():
        with jsonl_path.open('r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    data.append(json.loads(line))
    json_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')


In [6]:
def _scrape_doacoes_by_year_impl(
    year: int,
    jsonl_path: Path,
    csv_path: Path,
    checkpoint_path: Path,
    final_json_path: Path,
    headless: bool = True,
    max_pages: int | None = None,
    resume: bool = True,
) -> dict[str, Any]:
    jsonl_path.parent.mkdir(parents=True, exist_ok=True)

    if not resume:
        reset_outputs(jsonl_path, csv_path, checkpoint_path, final_json_path)

    checkpoint = load_checkpoint(checkpoint_path) if resume else {'visited_pages': [], 'total_rows': 0}
    visited_pages = {int(p) for p in checkpoint.get('visited_pages', [])}
    total_rows = int(checkpoint.get('total_rows', 0))

    resume_from_page: int | None = None
    if resume and visited_pages:
        resume_from_page = int(checkpoint.get('last_page') or max(visited_pages))
        if resume_from_page in visited_pages:
            visited_pages.remove(resume_from_page)
        print(f'[RESUME] retomando com sobreposicao de 1 pagina a partir da pagina {resume_from_page}.')

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=headless)
        page = browser.new_page()
        page.goto(URL, wait_until='domcontentloaded', timeout=120000)
        page.wait_for_selector(TABLE_SELECTOR, timeout=120000)
        page.wait_for_selector(SELECTED_PAGE_SELECTOR, timeout=120000)

        available_years = get_available_years(page)
        print('Anos disponiveis:', available_years)
        select_year(page, year)

        if resume_from_page and resume_from_page > 1:
            jumped = go_to_page(page, resume_from_page)
            if jumped:
                print(f'[RESUME] pagina inicial posicionada em {resume_from_page}.')
            else:
                print(f'[RESUME] nao foi possivel saltar diretamente para {resume_from_page}; continua pelo fluxo normal.')

        headers = get_headers(page)
        if not headers:
            browser.close()
            raise RuntimeError('Nao foram encontrados cabecalhos da tabela.')

        fieldnames = ['ano', 'pagina', 'linha_pagina'] + headers
        ensure_csv_header(csv_path, fieldnames)

        pages_scraped_this_run = 0
        safety_limit = (max_pages or 100000) + 500
        steps = 0

        while steps < safety_limit:
            steps += 1
            current_page = get_selected_page(page)

            if current_page not in visited_pages:
                rows = extract_rows(page, headers, year, current_page)
                append_jsonl(jsonl_path, rows)
                append_csv(csv_path, rows, fieldnames)

                visited_pages.add(current_page)
                pages_scraped_this_run += 1
                total_rows += len(rows)

                checkpoint_payload = {
                    'ano': year,
                    'visited_pages': sorted(visited_pages),
                    'total_rows': total_rows,
                    'last_page': current_page,
                    'updated_at_utc': datetime.now(timezone.utc).isoformat(),
                }
                save_checkpoint(checkpoint_path, checkpoint_payload)

                print(f'[OK] ano={year} pagina={current_page} +{len(rows)} linhas | total={total_rows}')
            else:
                print(f'[SKIP] pagina={current_page} ja gravada no checkpoint')

            if max_pages is not None and pages_scraped_this_run >= max_pages:
                print('Limite max_pages atingido para esta execucao.')
                break

            next_page = current_page + 1
            moved = click_page_number(page, next_page)
            if not moved:
                moved = click_forward_ellipsis(page, current_page)
            if not moved:
                print('Fim da paginacao.')
                break

            new_page = get_selected_page(page)
            if new_page in visited_pages and new_page <= current_page:
                print('Paragem de seguranca para evitar loop.')
                break

        browser.close()

    export_json_array_from_jsonl(jsonl_path, final_json_path)

    result = {
        'ano': year,
        'total_rows': total_rows,
        'paginas_gravadas': len(visited_pages),
        'jsonl': str(jsonl_path.resolve()),
        'csv': str(csv_path.resolve()),
        'checkpoint': str(checkpoint_path.resolve()),
        'json_final': str(final_json_path.resolve()),
    }
    print('Resumo final:', result)
    return result


def scrape_doacoes_by_year(
    year: int,
    jsonl_path: Path,
    csv_path: Path,
    checkpoint_path: Path,
    final_json_path: Path,
    headless: bool = True,
    max_pages: int | None = None,
    resume: bool = True,
) -> dict[str, Any]:
    import asyncio
    from concurrent.futures import ThreadPoolExecutor

    args = {
        'year': year,
        'jsonl_path': jsonl_path,
        'csv_path': csv_path,
        'checkpoint_path': checkpoint_path,
        'final_json_path': final_json_path,
        'headless': headless,
        'max_pages': max_pages,
        'resume': resume,
    }

    try:
        running_loop = asyncio.get_running_loop()
    except RuntimeError:
        running_loop = None

    if running_loop and running_loop.is_running():
        with ThreadPoolExecutor(max_workers=1) as executor:
            return executor.submit(_scrape_doacoes_by_year_impl, **args).result()

    return _scrape_doacoes_by_year_impl(**args)


In [7]:
import asyncio

result = await asyncio.to_thread(
    scrape_doacoes_by_year,
    year=YEAR,
    jsonl_path=JSONL_PATH,
    csv_path=CSV_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    final_json_path=FINAL_JSON_PATH,
    headless=HEADLESS,
    max_pages=MAX_PAGES,
    resume=RESUME,
)
result


[RESUME] retomando com sobreposicao de 1 pagina a partir da pagina 2738.
Anos disponiveis: [2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016, 2015, 2014, 2013]
[RESUME] a navegar... pagina atual=1, alvo=2738
[RESUME] salto: 1 -> 20 via ultimo visivel (20)
[RESUME] salto: 20 -> 21 via reticencias (...)
[RESUME] salto: 21 -> 31 via ultimo visivel (31)
[RESUME] salto: 31 -> 41 via ultimo visivel (41)
[RESUME] salto: 41 -> 51 via ultimo visivel (51)
[RESUME] salto: 51 -> 61 via ultimo visivel (61)
[RESUME] salto: 61 -> 71 via ultimo visivel (71)
[RESUME] salto: 71 -> 81 via ultimo visivel (81)
[RESUME] salto: 81 -> 91 via ultimo visivel (91)
[RESUME] salto: 91 -> 101 via ultimo visivel (101)
[RESUME] salto: 101 -> 111 via ultimo visivel (111)
[RESUME] salto: 111 -> 121 via ultimo visivel (121)
[RESUME] salto: 121 -> 131 via ultimo visivel (131)
[RESUME] salto: 131 -> 141 via ultimo visivel (141)
[RESUME] salto: 141 -> 151 via ultimo visivel (151)
[RESUME] salto: 151 -> 161 

{'ano': 2025,
 'total_rows': 69039,
 'paginas_gravadas': 3449,
 'jsonl': '/home/pedro/Projetos/Web_Scraping/scrapper/infarmed_output/doacoes_2025.jsonl',
 'csv': '/home/pedro/Projetos/Web_Scraping/scrapper/infarmed_output/doacoes_2025.csv',
 'checkpoint': '/home/pedro/Projetos/Web_Scraping/scrapper/infarmed_output/doacoes_2025_checkpoint.json',
 'json_final': '/home/pedro/Projetos/Web_Scraping/scrapper/infarmed_output/doacoes_2025.json'}

In [9]:
# Validacao rapida
with FINAL_JSON_PATH.open('r', encoding='utf-8') as f:
    data = json.load(f)

print('Total no JSON final:', len(data))
print('Exemplo de registo:')
data[0] if data else {}


Total no JSON final: 69039
Exemplo de registo:


{'ano': 2025,
 'pagina': 1,
 'linha_pagina': 1,
 'Nome Entidade Contribuinte': 'Boehringer Ingelheim Portugal, Lda.',
 'Tipo Declaração': 'Medicamentos',
 'Evento/Bem/Ação': 'Meeting in the box',
 'Quantia (€)': '600,00',
 'Nome Entidade Recetora': 'Nuno Capela Lda',
 'Estado': 'Por Validar'}